# Batch Abstract-Only Extraction Evaluation

Evaluate abstract-only metadata extraction across all Fuster validation samples (Dryad + Zenodo + Semantic Scholar).

**Approach:**
- Load validated ground truth from `data/dataset_092624_validated.xlsx`
- Join with original `data/dataset_092624.xlsx` to get abstract text (`full_text` column)
- Filter to records with non-null abstracts
- Run GPT-5-mini extraction using `text_classification_flow()` (Prefect, parallel)
- Compare against manually annotated ground truth
- Per-field P/R/F1 for all 14 evaluatable fields (8 core + 6 modulators)
- Segmented by source: Dryad vs Zenodo vs Semantic Scholar

**Cache note:** The joblib cache in `gpt_classify.py` uses `./cache` relative to the working directory.
This notebook ensures `os.chdir(PROJECT_ROOT)` is called before imports so the cache lands at
`{PROJECT_ROOT}/cache/` — which is NOT matched by the `*/cache/*` gitignore pattern and can be
committed and synced to your local machine.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

# ── Cache & working directory ──────────────────────────────────────────────
# IMPORTANT: set PROJECT_ROOT and chdir BEFORE importing any llm_metadata
# modules. gpt_classify.py creates Memory('./cache') at import time, so the
# resolved cache path depends on cwd at import time.
#
# Root-level cache/ is NOT matched by the '*/cache/*' gitignore pattern
# (that pattern needs at least one segment before 'cache'), so the cache at
# PROJECT_ROOT/cache/ can be committed and pulled locally.
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
os.chdir(PROJECT_ROOT)
print(f"Working directory set to: {PROJECT_ROOT}")
print(f"Joblib cache will land at: {PROJECT_ROOT / 'cache'}")

from datetime import datetime
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv()

# ── Project modules ────────────────────────────────────────────────────────
from llm_metadata.gpt_classify import SYSTEM_MESSAGE
from llm_metadata.text_pipeline import (
    TextClassificationConfig,
    TextInputRecord,
    TextOutputRecord,
    text_classification_flow,
    save_text_manifest,
)
from llm_metadata.schemas.fuster_features import (
    DatasetFeatures,
    DatasetFeaturesNormalized,
)
from llm_metadata.groundtruth_eval import (
    evaluate_indexed,
    EvaluationConfig,
    FuzzyMatchConfig,
    micro_average,
    macro_f1,
)

# ── Paths ──────────────────────────────────────────────────────────────────
VALIDATED_PATH = PROJECT_ROOT / "data/dataset_092624_validated.xlsx"
ORIGINAL_PATH  = PROJECT_ROOT / "data/dataset_092624.xlsx"
OUTPUT_DIR     = PROJECT_ROOT / "artifacts/abstract_results"

print(f"Validated data: {VALIDATED_PATH.exists()}")
print(f"Original data:  {ORIGINAL_PATH.exists()}")

## Step 1: Load Data

Load the validated ground truth and join with the original file to get abstract text.

In [ ]:
# Load validated ground truth (schema-valid, valid_yn='yes', all sources)
validated_df = pd.read_excel(VALIDATED_PATH)
print(f"Validated records: {len(validated_df)}")
print(f"By source:")
print(validated_df['source'].value_counts())
print(f"\nColumns: {list(validated_df.columns)}")

In [ ]:
# Load original file to get full_text (abstract) column
original_df = pd.read_excel(ORIGINAL_PATH, usecols=['id', 'full_text'])
original_df = original_df.rename(columns={'full_text': 'abstract'})
print(f"Original records: {len(original_df)}, with abstract: {original_df['abstract'].notna().sum()}")

# Merge on id — validated_df.id is the row ID from the original xlsx
validated_df['id'] = validated_df['id'].astype(int)
original_df['id'] = original_df['id'].astype(int)
gt_df = validated_df.merge(original_df, on='id', how='left')

print(f"\nAfter join: {len(gt_df)} rows")
print(f"With abstract: {gt_df['abstract'].notna().sum()}")
print(f"\nAbstract coverage by source:")
print(gt_df.groupby('source')['abstract'].apply(lambda s: f"{s.notna().sum()}/{len(s)}"))

In [ ]:
# Filter to records with abstracts
with_abstract_df = gt_df[gt_df['abstract'].notna()].copy()
print(f"Records with abstract: {len(with_abstract_df)}")
print(f"By source:")
print(with_abstract_df['source'].value_counts())

# Validate ground truth through DatasetFeaturesNormalized
RELEVANT_COLS = [
    'data_type', 'geospatial_info_dataset', 'spatial_range_km2',
    'temporal_range', 'temp_range_i', 'temp_range_f',
    'species', 'referred_dataset',
    'time_series', 'multispecies', 'threatened_species',
    'new_species_science', 'new_species_region', 'bias_north_south',
]

ground_truth_by_id = {}  # keyed by record id (str)
validation_errors = []

for _, row in with_abstract_df.iterrows():
    record_id = str(row['id'])
    try:
        row_dict = {col: row[col] for col in RELEVANT_COLS if col in row.index}
        validated = DatasetFeaturesNormalized.model_validate(row_dict)
        ground_truth_by_id[record_id] = validated
    except Exception as e:
        validation_errors.append({'id': record_id, 'error': str(e)})

print(f"\nGround truth validated: {len(ground_truth_by_id)}")
print(f"Validation errors: {len(validation_errors)}")
if validation_errors:
    for err in validation_errors[:3]:
        print(f"  id={err['id']}: {err['error'][:120]}")

## Step 2: Configure Pipeline

Set up abstract text classification with `DatasetFeatures` schema and `SYSTEM_MESSAGE`.

In [ ]:
config = TextClassificationConfig(
    model="gpt-5-mini",
    reasoning={"effort": "low"},
    max_output_tokens=4096,
    text_format=DatasetFeatures,
    system_message=SYSTEM_MESSAGE,
    max_workers=5,
    output_dir=OUTPUT_DIR,
)

print("Abstract Classification Configuration:")
print(f"  Model:         {config.model}")
print(f"  Reasoning:     {config.reasoning}")
print(f"  Schema:        {config.text_format.__name__}")
print(f"  Max workers:   {config.max_workers}")
print(f"\nSystem prompt preview (first 200 chars):")
print(f"  {config.system_message[:200]}...")

## Step 3: Extract

Build `TextInputRecord` list and run `text_classification_flow()` (Prefect, ThreadPool).
Joblib cache ensures reruns are free — cached results are reused automatically.

In [ ]:
# Build input records — id = str(row id), text = abstract
input_records = []
for _, row in with_abstract_df.iterrows():
    input_records.append(TextInputRecord(
        id=str(int(row['id'])),
        text=str(row['abstract']),
        metadata={
            'source': row['source'],
            'source_url': row.get('source_url', None),
        }
    ))

print(f"Input records: {len(input_records)}")
print(f"Sample record id: {input_records[0].id}")
print(f"Sample text (first 200 chars): {input_records[0].text[:200]}")

In [ ]:
# Run batch extraction
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_manifest_path = OUTPUT_DIR / f"abstract_results_{timestamp}.csv"

print(f"Running abstract classification on {len(input_records)} records...")
print(f"Model: {config.model}, reasoning: {config.reasoning}")
print(f"Output manifest: {output_manifest_path}")
print()

results = text_classification_flow(
    input_records=input_records,
    config=config,
    output_manifest=output_manifest_path,
)

success_count = sum(1 for r in results if r.status == "success")
error_count   = sum(1 for r in results if r.status == "error")
print(f"\nProcessing complete: {success_count} success, {error_count} errors")

## Step 4: Prepare Extractions for Evaluation

Convert extraction results to `DatasetFeaturesNormalized` Pydantic models.

In [ ]:
# Build predictions keyed by record id
predictions_by_id = {}

for result in results:
    if result.status != "success" or not result.extraction:
        continue
    record_id = result.id
    try:
        # Validate prediction through normalized schema for fair comparison
        pred = DatasetFeaturesNormalized.model_validate(result.extraction)
        predictions_by_id[record_id] = pred
    except Exception as e:
        print(f"Validation error for id={record_id}: {e}")

print(f"Valid predictions: {len(predictions_by_id)}")

# Find common IDs for evaluation
common_ids = set(predictions_by_id.keys()) & set(ground_truth_by_id.keys())
print(f"Common IDs for evaluation: {len(common_ids)}")

# Source breakdown for evaluated records
id_to_source = dict(zip(with_abstract_df['id'].astype(str), with_abstract_df['source']))
source_counts = {}
for rid in common_ids:
    src = id_to_source.get(rid, 'unknown')
    source_counts[src] = source_counts.get(src, 0) + 1
print(f"\nBy source:")
for src, cnt in sorted(source_counts.items()):
    print(f"  {src}: {cnt}")

## Step 5: Evaluate

Compute per-field P/R/F1 for all 14 evaluatable fields: 8 core EBV fields + 6 modulator booleans.

In [ ]:
# Fields to evaluate
CORE_FIELDS = [
    'data_type', 'geospatial_info_dataset', 'spatial_range_km2',
    'temporal_range', 'temp_range_i', 'temp_range_f',
    'species', 'referred_dataset',
]
MODULATOR_FIELDS = [
    'time_series', 'multispecies', 'threatened_species',
    'new_species_science', 'new_species_region', 'bias_north_south',
]
EVAL_FIELDS = CORE_FIELDS + MODULATOR_FIELDS

# Evaluation config: case-insensitive, set-based lists, fuzzy species matching
eval_config = EvaluationConfig(
    casefold_strings=True,
    treat_lists_as_sets=True,
    enhanced_species_matching=True,
    enhanced_species_threshold=70,
)

true_by_id = {rid: ground_truth_by_id[rid] for rid in common_ids}
pred_by_id = {rid: predictions_by_id[rid] for rid in common_ids}

abstract_report = evaluate_indexed(
    true_by_id=true_by_id,
    pred_by_id=pred_by_id,
    fields=EVAL_FIELDS,
    config=eval_config,
)

print("ABSTRACT-ONLY Extraction Metrics (all sources):")
print("=" * 70)
display(abstract_report.metrics_to_pandas())

In [ ]:
# Aggregate metrics
abstract_micro = micro_average(abstract_report.field_metrics.values())
abstract_macro = macro_f1(abstract_report.field_metrics.values())

# Split by field group
core_metrics = {k: v for k, v in abstract_report.field_metrics.items() if k in CORE_FIELDS}
mod_metrics  = {k: v for k, v in abstract_report.field_metrics.items() if k in MODULATOR_FIELDS}
core_micro = micro_average(core_metrics.values())
mod_micro  = micro_average(mod_metrics.values())

print("\nAggregate Metrics:")
print("=" * 55)
print(f"{'Metric':<30} {'Value':>12}")
print("-" * 55)
print(f"{'[ALL] Micro Precision':<30} {abstract_micro.precision or 0:>12.3f}")
print(f"{'[ALL] Micro Recall':<30} {abstract_micro.recall or 0:>12.3f}")
print(f"{'[ALL] Micro F1':<30} {abstract_micro.f1 or 0:>12.3f}")
print(f"{'[ALL] Macro F1':<30} {abstract_macro or 0:>12.3f}")
print("-" * 55)
print(f"{'[CORE] Micro F1':<30} {core_micro.f1 or 0:>12.3f}")
print(f"{'[MODULATOR] Micro F1':<30} {mod_micro.f1 or 0:>12.3f}")
print("-" * 55)
print(f"{'Records Evaluated':<30} {len(common_ids):>12}")
print("=" * 55)

## Step 6: Analysis

Per-field metrics table, segmentation by source (Dryad / Zenodo / Semantic Scholar), and mismatch examples.

In [ ]:
# Per-field metrics table
field_data = []
for f in EVAL_FIELDS:
    fm = abstract_report.field_metrics.get(f)
    if fm:
        field_data.append({
            'field': f,
            'group': 'core' if f in CORE_FIELDS else 'modulator',
            'precision': round(fm.precision or 0, 3),
            'recall':    round(fm.recall or 0, 3),
            'f1':        round(fm.f1 or 0, 3),
            'tp': fm.tp, 'fp': fm.fp, 'fn': fm.fn,
            'exact_match_rate': round(fm.exact_match_rate or 0, 3),
        })

field_df = pd.DataFrame(field_data)
print("Per-Field Metrics (Abstract-Only):")
display(field_df)

if field_data:
    best  = max(field_data, key=lambda x: x['f1'])
    worst = min(field_data, key=lambda x: x['f1'])
    print(f"\nBest:  {best['field']}  (F1={best['f1']})")
    print(f"Worst: {worst['field']} (F1={worst['f1']})")

In [ ]:
# Per-source evaluation
SOURCES = ['dryad', 'zenodo', 'semantic_scholar']

source_summary = []

for src in SOURCES:
    src_ids = {rid for rid in common_ids if id_to_source.get(rid) == src}
    if not src_ids:
        continue

    src_true = {rid: true_by_id[rid] for rid in src_ids}
    src_pred = {rid: pred_by_id[rid] for rid in src_ids}

    src_report = evaluate_indexed(
        true_by_id=src_true,
        pred_by_id=src_pred,
        fields=EVAL_FIELDS,
        config=eval_config,
    )

    src_micro = micro_average(src_report.field_metrics.values())
    src_macro = macro_f1(src_report.field_metrics.values())

    source_summary.append({
        'source': src,
        'n': len(src_ids),
        'micro_precision': round(src_micro.precision or 0, 3),
        'micro_recall':    round(src_micro.recall or 0, 3),
        'micro_f1':        round(src_micro.f1 or 0, 3),
        'macro_f1':        round(src_macro or 0, 3),
    })

    print(f"\n{'='*60}")
    print(f"SOURCE: {src.upper()} (n={len(src_ids)})")
    print(f"{'='*60}")
    display(src_report.metrics_to_pandas())

print("\n" + "="*60)
print("CROSS-SOURCE SUMMARY")
print("="*60)
display(pd.DataFrame(source_summary))

In [ ]:
# Mismatch examples (for presentation)
results_detail_df = abstract_report.to_pandas()
mismatches = results_detail_df[~results_detail_df['match']]
print(f"Total mismatches: {len(mismatches)}")

# Show a sample of mismatches per field
print("\nMismatch count by field:")
print(mismatches['field'].value_counts())

print("\nSample mismatches (first 15):")
display(mismatches[['record_id', 'field', 'true_value', 'pred_value', 'tp', 'fp', 'fn']].head(15))

## Step 7: Cost Analysis

In [ ]:
successful_results = [r for r in results if r.status == "success"]

if successful_results:
    total_cost         = sum(r.cost_usd or 0 for r in successful_results)
    total_input_tokens = sum(r.input_tokens or 0 for r in successful_results)
    total_output_tokens= sum(r.output_tokens or 0 for r in successful_results)

    print("\nCOST ANALYSIS (Abstract-Only Extraction)")
    print("=" * 55)
    print(f"{'Metric':<35} {'Value':>18}")
    print("-" * 55)
    print(f"{'Records Processed':<35} {len(successful_results):>18}")
    print(f"{'Avg Input Tokens / record':<35} {total_input_tokens / len(successful_results):>18,.0f}")
    print(f"{'Avg Output Tokens / record':<35} {total_output_tokens / len(successful_results):>18,.0f}")
    print("-" * 55)
    print(f"{'Total Input Tokens':<35} {total_input_tokens:>18,}")
    print(f"{'Total Output Tokens':<35} {total_output_tokens:>18,}")
    print(f"{'Total Cost (USD)':<35} ${total_cost:>17.4f}")
    print(f"{'Avg Cost per record (USD)':<35} ${total_cost / len(successful_results):>17.5f}")
    print("=" * 55)

    # Per-source cost
    print("\nCost breakdown by source:")
    id_results = {r.id: r for r in successful_results}
    for src in SOURCES:
        src_ids = {rid for rid in common_ids if id_to_source.get(rid) == src}
        src_results = [id_results[rid] for rid in src_ids if rid in id_results]
        if src_results:
            src_cost = sum(r.cost_usd or 0 for r in src_results)
            print(f"  {src:<22}: {len(src_results):>4} records, ${src_cost:.4f} total, ${src_cost/len(src_results):.5f}/record")

## Step 8: Export

Save HTML report to `notebooks/results/` and extraction CSV for WU-D1 comparison.

In [ ]:
# Timestamped results directory
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
results_dir = PROJECT_ROOT / "notebooks" / "results" / f"batch_abstract_evaluation_{run_id}"
results_dir.mkdir(parents=True, exist_ok=True)

# ── Extraction CSV (for WU-D1 comparison) ─────────────────────────────────
# Build flat DataFrame: one row per (record_id, source, field, true, pred, match)
detail_df = abstract_report.to_pandas()
detail_df['source'] = detail_df['record_id'].map(id_to_source)
extraction_csv = results_dir / "abstract_evaluation_detail.csv"
detail_df.to_csv(extraction_csv, index=False)
print(f"Saved detail CSV: {extraction_csv}")

# Per-field summary CSV
field_df.to_csv(results_dir / "abstract_field_metrics.csv", index=False)

# Per-source summary CSV
pd.DataFrame(source_summary).to_csv(results_dir / "abstract_source_metrics.csv", index=False)

print(f"\nAll results saved to: {results_dir}")

In [ ]:
# HTML report via nbconvert (run after notebook is complete)
try:
    import subprocess, sys
    notebook_path = PROJECT_ROOT / "notebooks" / "batch_abstract_evaluation.ipynb"
    html_path = results_dir / "batch_abstract_evaluation.html"

    result = subprocess.run(
        [sys.executable, "-m", "nbconvert",
         "--to", "html",
         "--output", str(html_path),
         str(notebook_path)],
        capture_output=True, text=True
    )

    if result.returncode == 0:
        print(f"HTML report: {html_path}")
    else:
        print(f"nbconvert output: {result.stderr[:200]}")
        print(f"(Run nbconvert manually if needed)")
except Exception as e:
    print(f"Could not auto-generate HTML: {e}")
    print("Run manually: jupyter nbconvert --to html notebooks/batch_abstract_evaluation.ipynb")

## Summary

| Metric | Value |
|--------|-------|
| Records evaluated | — |
| Micro F1 (all fields) | — |
| Macro F1 (all fields) | — |
| Core fields Micro F1 | — |
| Modulator Micro F1 | — |
| Total cost (USD) | — |

*(Fill in after running)*